# Lab 05 — Your First Neural Network
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 5 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab05-first-neural-network/Lab05_First_Neural_Network.ipynb)

In Weeks 3–4 you trained a straight line, ŷ = w·x + b, with a cost function J and gradient descent. Today you stack many of those little models — **neurons** — into a **neural network**. You will watch one learn in your browser, compute one by hand, and then train one on 768 building designs to predict how much energy they need for heating. Questions marked 🏠 may be finished at home before the deadline; everything else is meant for lab time.

### What you will be able to do
1. Explain, from experiments in TensorFlow Playground, what hidden layers, activation functions, and the learning rate do.
2. Compute the forward pass of a small network (weighted sums, ReLU, output) by hand and with NumPy matrices, and count its parameters.
3. Train a neural network with scikit-learn's `MLPRegressor` and compare it with linear regression using train and test error.
4. Recognize overfitting from the gap between training and test error.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `building_energy_efficiency.csv` — 768 simulated building shapes with their annual heating and cooling loads (Tsanas & Xifara, 2012; CC BY 4.0). See the course `data/` folder.

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Watch a neural network learn: TensorFlow Playground (15 min)

**TensorFlow Playground** is a free website that trains a tiny neural network right in your browser. Nothing to
install, and you cannot break it. Open it in a **new browser tab** next to this notebook:
<https://playground.tensorflow.org>

**How to read the screen**
- **DATA** (left): the dots. Each dot has two inputs, **X₁** (horizontal position) and **X₂** (vertical position).
  Its color is the label: **blue** or **orange**. The task is *classification*: predict the color from (X₁, X₂).
- **FEATURES**: the inputs fed into the network. Today we use only X₁ and X₂.
- **HIDDEN LAYERS**: the `+` / `−` buttons add or remove layers, and the small `+` / `−` above each layer add or
  remove neurons. Each little square is one neuron, showing what it has learned.
- **OUTPUT** (right): the background color is the network's prediction everywhere on the map. **Test loss** and
  **Training loss** measure the error (lower is better), like the cost J from Week 3.
- Top bar: **▶** starts and pauses training, **Epoch** counts passes through the training data, and the
  **Learning rate** box is the step size α from Lab 04. **Activation** picks the function each hidden neuron applies.

Do **Experiments 1–3**; Experiment 4 and the challenges are optional. Each link below sets every option for you. Click it, press **▶**, let it run to about **Epoch 300**, then press
**▶** again to pause. (Playground reads the settings only when the page loads: if you paste a link into a tab that
already shows Playground and nothing changes, press the browser's reload button.) Write down the **test loss** and describe the picture. Results change slightly from run to run
because the starting weights are random — that is normal.

### Experiment 1 — No hidden layers
[Open Experiment 1](https://playground.tensorflow.org/#activation=relu&learningRate=0.03&networkShape=&dataset=circle&noise=0&percTrainData=50&batchSize=10&seed=0.42000&x=true&y=true&problem=classification) — circle data, **0 hidden layers**. The network is just
X₁ and X₂ feeding one output neuron: a weighted sum, very much like your Week 3 model.

**✍️ Question:** Test loss after about 300 epochs? What shape is the boundary between the blue and orange regions, and why can't this network separate the two classes?

*Your answer:* 

### Experiment 2 — One hidden layer with ReLU
[Open Experiment 2](https://playground.tensorflow.org/#activation=relu&learningRate=0.03&networkShape=4&dataset=circle&noise=0&percTrainData=50&batchSize=10&seed=0.42000&x=true&y=true&problem=classification) — the same data, now with **one hidden layer of 4 neurons**
and the **ReLU** activation. Hover your mouse over each hidden neuron to see the line *it* draws.

**✍️ Question:** Test loss after about 300 epochs? What shape is the boundary now? Looking at the hidden neurons, how does the network build that shape?

*Your answer:* 

### Experiment 3 — Take away the nonlinearity
[Open Experiment 3](https://playground.tensorflow.org/#activation=linear&learningRate=0.03&networkShape=4,4&dataset=circle&noise=0&percTrainData=50&batchSize=10&seed=0.42000&x=true&y=true&problem=classification) — now **two** hidden layers of 4 neurons, but the activation
is **Linear** (each neuron just passes its weighted sum through, no bending).

**✍️ Question:** What happens to the test loss and to the boundary? Explain why adding more layers did not help.

*Your answer:* 

### Optional: Experiment 4 — Learning rate: too big and too small
Same network as Experiment 2 (4 ReLU neurons), two different learning rates:
- [Learning rate **3** (too big)](https://playground.tensorflow.org/#activation=relu&learningRate=3&networkShape=4&dataset=circle&noise=0&percTrainData=50&batchSize=10&seed=0.42000&x=true&y=true&problem=classification)
- [Learning rate **0.0001** (too small)](https://playground.tensorflow.org/#activation=relu&learningRate=0.0001&networkShape=4&dataset=circle&noise=0&percTrainData=50&batchSize=10&seed=0.42000&x=true&y=true&problem=classification)

**✍️ Question:** (Optional) Describe how the test loss behaves in each case. Connect this to what you saw with α in Lab 04.

*Your answer:* 

### More optional challenges (if you finish early)
- **Hand-made features:** [0 hidden layers, but inputs X₁² and X₂² added](https://playground.tensorflow.org/#activation=relu&learningRate=0.03&networkShape=&dataset=circle&noise=0&percTrainData=50&batchSize=10&seed=0.42000&x=true&y=true&problem=classification&xSquared=true&ySquared=true).
  Why can this network solve the circle *without* any hidden layer? (Hint: the circle is x₁² + x₂² = r².)
- **Spiral:** choose the spiral dataset (bottom-right picture under DATA). How many layers and neurons do you need
  to get a test loss below 0.1?

## Part 2 — A forward pass by hand (20 min)

### 2.1 A neuron = your Week 3 model + an activation
A **neuron** computes a weighted sum of its inputs plus a bias — exactly your linear model — and then passes the
result through an **activation function** g:

$$z = w_1 x_1 + w_2 x_2 + b, \qquad h = g(z)$$

The most common activation today is **ReLU** (rectified linear unit): keep positive numbers, replace negative
numbers with 0. In symbols, ReLU(z) = max(0, z).

`np.maximum(0, z)` compares 0 with `z` and keeps the larger one. It works on a single number *and* on a whole
array at once.

> 💡 **Hint:** `np.maximum(0, z)` is the whole function body: `return np.maximum(0, z)`.

In [ ]:
# TODO: finish the ReLU function. Return z when z > 0, and 0 otherwise.
def relu(z):
    return None   # replace None (use np.maximum)

print(relu(-1.5), relu(1.0))

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert relu(1.0) is not None, "relu returns None: replace `None` with np.maximum(0, z)."
assert relu(-1.5) == 0 and relu(1.0) == 1.0 and relu(0.0) == 0, "relu(-1.5) should be 0 and relu(1.0) should be 1.0: keep positives, replace negatives with 0."
assert np.allclose(relu(np.array([-2.0, 0.5, 3.0])), [0.0, 0.5, 3.0]), "relu should also work on arrays: use np.maximum, not an if statement."
print("Correct!")

Run the next cell to compare ReLU with two older activation functions you saw in lecture, **sigmoid** (squashes any number into 0 to 1) and **tanh** (squashes into −1 to 1).

In [ ]:
z = np.linspace(-4, 4, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(z, relu(z), label="ReLU: max(0, z)", color="#AB0520", linewidth=2.5)
plt.plot(z, 1 / (1 + np.exp(-z)), label="sigmoid: 1 / (1 + e^(-z))", color="#0C234B")
plt.plot(z, np.tanh(z), label="tanh(z)", color="#378DBD", linestyle="--")
plt.axhline(0, color="gray", linewidth=0.5)
plt.xlabel("weighted sum z (no units)")
plt.ylabel("neuron output h = g(z)")
plt.title("Three common activation functions")
plt.legend()
plt.show()

### 2.2 The network from lecture: 2 inputs → 2 hidden neurons (ReLU) → 1 output
This is the same tiny network as on the Week 5 slides. Inputs: **x₁ = 1.0, x₂ = 2.0**.

| Neuron | weight on 1st input | weight on 2nd input | bias | activation |
|---|---|---|---|---|
| hidden neuron 1 | 0.5 (on x₁) | −1.0 (on x₂) | 0 | ReLU |
| hidden neuron 2 | 1.0 (on x₁) | 0.5 (on x₂) | −1 | ReLU |
| output neuron | 1.0 (on h₁) | −2.0 (on h₂) | 0.5 | none (linear) |

A **forward pass** means: compute each hidden neuron (weighted sum z, then ReLU → h), then feed h₁ and h₂ into
the output neuron to get the prediction ŷ. Lists from Lab 01 hold each neuron's weights: `w_h1[0]` is the weight on
x₁ and `w_h1[1]` is the weight on x₂.

In [ ]:
x1, x2 = 1.0, 2.0          # inputs (already scaled, so no units)

w_h1, b_h1 = [0.5, -1.0], 0.0     # hidden neuron 1: weights on (x1, x2), bias
w_h2, b_h2 = [1.0, 0.5], -1.0     # hidden neuron 2
w_out, b_out = [1.0, -2.0], 0.5   # output neuron: weights on (h1, h2), bias

> 💡 **Hint:** Copy the pattern in the comments. For h1 and h2, call your `relu` function. Check by hand: z1 = 0.5·1 + (−1.0)·2 + 0.

In [ ]:
# TODO: compute the forward pass one step at a time.
z1 = None      # weighted sum of hidden neuron 1: w_h1[0]*x1 + w_h1[1]*x2 + b_h1
h1 = None      # ReLU of z1
z2 = None      # weighted sum of hidden neuron 2
h2 = None      # ReLU of z2
y_hat = None   # output neuron (linear, no ReLU): w_out[0]*h1 + w_out[1]*h2 + b_out

print(f"z1 = {z1}, h1 = {h1}")
print(f"z2 = {z2}, h2 = {h2}")
print(f"y_hat = {y_hat}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert None not in (z1, h1, z2, h2, y_hat), "Replace every None in the cell above."
assert abs(z1 - (-1.5)) < 1e-9, "z1 should be 0.5*1.0 + (-1.0)*2.0 + 0 = -1.5. Check that you used w_h1 and b_h1."
assert abs(h1 - 0.0) < 1e-9, "h1 = relu(z1), and z1 is negative, so h1 should be 0."
assert abs(z2 - 1.0) < 1e-9, "z2 should be 1.0*1.0 + 0.5*2.0 + (-1.0) = 1.0. Did you copy neuron 1's weights or bias by mistake?"
assert abs(h2 - 1.0) < 1e-9, "h2 = relu(z2) = 1.0."
assert abs(y_hat - (-1.5)) < 1e-9, "y_hat = 1.0*h1 + (-2.0)*h2 + 0.5 = -1.5. The output neuron uses h1 and h2 (not x1, x2) and no ReLU."
print("Correct! z1 = -1.5, h1 = 0, z2 = 1.0, h2 = 1.0, y_hat = -1.5")

**✍️ Question:** Hidden neuron 1 outputs h₁ = 0 for this input. What does that mean? If you changed the output weight on h₁ (1.0) to any other number, would ŷ change for this input?

*Your answer:* 

### 2.3 The same thing with matrices
Real networks have thousands of neurons, so we stack all the weights of a layer into a **matrix** (a 2-D array):
row *i* holds the weights of neuron *i*. The `@` operator multiplies a matrix by a vector: it takes each row, multiplies
it number-by-number with the vector, and adds up — exactly the weighted sums you just wrote, all at once.

`.shape` tells you the size of an array as (rows, columns).

In [ ]:
x  = np.array([1.0, 2.0])          # input vector
W1 = np.array([[0.5, -1.0],         # row 1 = weights of hidden neuron 1
               [1.0,  0.5]])        # row 2 = weights of hidden neuron 2
b1 = np.array([0.0, -1.0])          # hidden-layer biases
W2 = np.array([[1.0, -2.0]])        # output layer: 1 neuron (1 row), 2 weights (2 columns)
b2 = np.array([0.5])                # output bias

print("W1 has shape", W1.shape, "and W2 has shape", W2.shape)
print("W1 @ x =", W1 @ x, " <- the two weighted sums before adding the biases")

> 💡 **Hint:** `z_hidden = W1 @ x + b1`, then `h = relu(z_hidden)`, then the same pattern with W2, h, b2.

In [ ]:
# TODO: the whole forward pass in three lines.
z_hidden = None    # W1 @ x + b1
h = None           # ReLU of z_hidden
y_hat_vec = None   # W2 @ h + b2

print("z_hidden =", z_hidden, " h =", h, " y_hat =", y_hat_vec)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert z_hidden is not None and h is not None and y_hat_vec is not None, "Replace every None in the cell above."
assert np.allclose(z_hidden, [-1.5, 1.0]), "z_hidden = W1 @ x + b1 should be [-1.5, 1.0], the same z1 and z2 as before."
assert np.allclose(h, [0.0, 1.0]), "h = relu(z_hidden) should be [0, 1]."
assert np.allclose(y_hat_vec, [-1.5]), "y_hat = W2 @ h + b2 should be [-1.5]. Use h (not x) in the output layer."
print("Correct! Same answer as the step-by-step version.")

### 2.4 Counting parameters
The **parameters** are all the numbers training can change: every weight and every bias. `.size` gives the number
of entries in an array.

> 💡 **Hint:** Add the four sizes together. You should get 4 + 2 + 2 + 1.

In [ ]:
# TODO: total number of parameters (weights + biases) in this network.
n_params = None   # use W1.size, b1.size, W2.size, b2.size
print("Parameters:", n_params)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert n_params is not None, "Replace None with the sum of the four sizes."
assert n_params == 9, "Expected 9: W1 has 4 weights, b1 has 2 biases, W2 has 2 weights, b2 has 1 bias."
print("Correct!")

A layer with `n_in` inputs and `n_out` neurons has `n_in × n_out` weights plus `n_out` biases. The
function below applies that rule layer by layer. `zip(sizes[:-1], sizes[1:])` pairs each layer size with the next
one: for `[2, 2, 1]` it gives the pairs (2, 2) and (2, 1).

In [ ]:
def count_params(sizes):
    """Weights + biases of a fully connected network with layer sizes like [2, 2, 1]."""
    total = 0
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        total += n_in * n_out + n_out     # weights + biases of one layer
    return total

print("[2, 2, 1]  ->", count_params([2, 2, 1]), "parameters")

**More than one output.** A network that predicts heating *and* cooling load has **2** output neurons.
Each output neuron has one weight per hidden neuron plus its own bias. For 8 inputs → 10 hidden → 2 outputs: the
hidden layer has 8 × 10 + 10 = 90 parameters and the output layer 10 × **2** + **2** = 22, so 112 in total.

In [ ]:
print("[8, 10, 2] ->", count_params([8, 10, 2]), "parameters")

**✍️ Question:** In Part 3 you will train a network with **8 inputs → 10 hidden neurons → 1 output**. Predict its number of parameters **by hand** first (show the arithmetic), then check with `count_params([8, 10, 1])` in a new code cell.

*Your answer:* 

In [ ]:
# Your check here: run count_params([8, 10, 1])

### 2.5 Why the activation matters (Experiment 3 again, with numbers)
What if we delete ReLU? Then the hidden layer just passes z through. Two linear layers can always be merged into
**one** linear layer: W = W2 @ W1 and b = W2 @ b1 + b2. Run the cell.

In [ ]:
y_no_relu = W2 @ (W1 @ x + b1) + b2          # two layers, no activation
W_comb = W2 @ W1                              # merged weights
b_comb = W2 @ b1 + b2                         # merged bias
print("Two linear layers:", y_no_relu)
print("One merged layer :", W_comb @ x + b_comb, " with weights", W_comb, "and bias", b_comb)

**✍️ Question:** 🏠 *Finish at home if you run short of time.* Without ReLU the prediction changes from −1.5 to −3.0, and the two layers collapse into one line ŷ = −1.5·x₁ − 2.0·x₂ + 2.5. In one or two sentences: why does a network need a nonlinear activation function?

*Your answer:* 

## Part 3 — Train a neural network on building energy (30 min)

### 3.1 The data and the question
`building_energy_efficiency.csv` has 768 building shapes simulated with energy software. All have the same volume
and materials; only the **shape, glazing (windows), and orientation** change. Can we predict the yearly
**heating load** (kWh per m² of floor area) from the eight design inputs? A quick, accurate model like this — a
**surrogate model** — lets an architect compare many early design options without running a full simulation each
time.

| Input | Meaning |
|---|---|
| `relative_compactness` | how compact the shape is (1 = most compact) |
| `surface_area_m2`, `wall_area_m2`, `roof_area_m2` | envelope areas (m²) |
| `overall_height_m` | 3.5 m or 7 m |
| `orientation` | 2 = N, 3 = E, 4 = S, 5 = W (a category code) |
| `glazing_area_ratio` | window area / floor area (0 to 0.4) |
| `glazing_distribution` | how windows are spread over the façades (a category code) |

We treat the two category codes as plain numbers to keep things simple.

In [ ]:
df = pd.read_csv(get_data("building_energy_efficiency.csv"))
print(df.shape)
df.head()

You know the next steps from Lab 04: choose **X** and **y**, split into **training** and **test** sets,
and **scale** the inputs (neural networks train much better when every input has mean 0 and standard deviation 1).
The scaler learns its means and standard deviations from the **training** data only, so the test set stays truly
unseen.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ["relative_compactness", "surface_area_m2", "wall_area_m2", "roof_area_m2",
            "overall_height_m", "orientation", "glazing_area_ratio", "glazing_distribution"]
X = df[features]
y = df["heating_load_kwh_m2"]          # kWh/m² per year

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)   # learn mean and std from the training buildings only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print(len(X_train), "training buildings,", len(X_test), "test buildings")
print(f"Heating load ranges from {y.min():.1f} to {y.max():.1f} kWh/m² (mean {y.mean():.1f})")

This helper prints the **mean absolute error (MAE)** on the training and test sets, and the test **R²** — the same metrics as Lab 04.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

def evaluate(model, name):
    """Print and return the train and test MAE (kWh/m²) of a fitted model."""
    mae_train = mean_absolute_error(y_train, model.predict(X_train_s))
    mae_test = mean_absolute_error(y_test, model.predict(X_test_s))
    r2_test = r2_score(y_test, model.predict(X_test_s))
    print(f"{name:>20}: train MAE = {mae_train:.2f}, test MAE = {mae_test:.2f} kWh/m²,  test R² = {r2_test:.3f}")
    return mae_train, mae_test

### 3.2 Baseline: linear regression
Always start with a **baseline**: a simple reference model that any fancier model must beat. If the neural network cannot beat it, the network is not worth it.

> 💡 **Hint:** Same as Lab 04: `lin = LinearRegression()`, then `lin.fit(X_train_s, y_train)`, then `evaluate(lin, "Linear regression")`.

In [ ]:
from sklearn.linear_model import LinearRegression

# TODO: create a LinearRegression model, fit it on the SCALED training data, then call evaluate(lin, "Linear regression").
lin = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert lin is not None and hasattr(lin, "coef_"), "Fit the model with lin.fit(X_train_s, y_train)."
print("Linear baseline is ready.")

### 3.3 Your first neural network: `MLPRegressor`
**MLP** stands for *multi-layer perceptron* — the fully connected network from Part 2. The settings we use:

| Setting | Meaning |
|---|---|
| `hidden_layer_sizes=(10,)` | one hidden layer with 10 neurons. `(20, 20)` would be two hidden layers of 20. The comma in `(10,)` tells Python it is a list-like **tuple** with one number. |
| `activation="relu"` | the default, so we do not need to type it |
| `learning_rate_init=0.01` | the learning rate α (step size) |
| `max_iter=500` | at most 500 **epochs** (passes through the training data) |
| `random_state=42` | the random starting weights are the same every run, so your results match your classmates' |

Training uses **Adam**, an improved version of gradient descent that adjusts its step size as it goes, and it
updates the weights after each **mini-batch** of 200 buildings instead of after the whole data set. The idea is the
same as Lab 04: compute the cost, follow the slope downhill, repeat.

Sometimes scikit-learn warns that training "has not converged" when it stops at `max_iter`. We turn that warning
off below and check the learning curve ourselves instead.

> 💡 **Hint:** Replace `None` with `(10,)`. Then add `mlp.fit(X_train_s, y_train)` and `evaluate(mlp, "MLP (10,)")` — the same pattern as the linear model.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# TODO: set hidden_layer_sizes to ONE hidden layer of 10 neurons, then fit on the scaled training data and evaluate.
mlp = MLPRegressor(hidden_layer_sizes=None, learning_rate_init=0.01, max_iter=500, random_state=42)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert mlp.hidden_layer_sizes == (10,), "Use hidden_layer_sizes=(10,) (note the comma)."
assert hasattr(mlp, "loss_curve_"), "Fit the model with mlp.fit(X_train_s, y_train)."
print("Neural network trained in", mlp.n_iter_, "epochs.")

**✍️ Question:** Compare the test MAE of the neural network with the linear baseline. Which is better, and by roughly what percentage?

*Your answer:* 

### 3.4 The learning curve and the parameter count
`mlp.loss_curve_` stores the training cost after every epoch. scikit-learn's cost is ½ × the mean squared error —
the same **J** you used in Weeks 3–4 — plus a tiny extra term that discourages very large weights.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_, color="#0C234B")
plt.yscale("log")            # a log scale shows both the fast early drop and the slow later progress
plt.xlabel("Epoch")
plt.ylabel("Training cost J  ((kWh/m²)²)")
plt.title("Learning curve: MLP with one hidden layer of 10 neurons")
plt.grid(alpha=0.3)
plt.show()

`mlp.coefs_` is a list of the weight matrices (one per layer) and `mlp.intercepts_` a list of the bias vectors. Does the count match your hand calculation from Part 2?

In [ ]:
for i, (W, b) in enumerate(zip(mlp.coefs_, mlp.intercepts_), start=1):
    print(f"layer {i}: weights {W.shape}, biases {b.shape}")
n_mlp = sum(W.size for W in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print("Total parameters:", n_mlp)

**✍️ Question:** 🏠 *Finish at home if you run short of time.* Describe the learning curve. Why does it drop quickly at first and then flatten?

*Your answer:* 

### 3.5 Bigger or smaller networks?
The loop below trains one network for each architecture in the list and collects the results in a table.

> 💡 **Hint:** Put your tuple inside the empty brackets, e.g. `architectures + [(100,)]`.

In [ ]:
architectures = [(1,), (3,), (10,), (50,), (20, 20)]
# TODO: add one architecture of your own to the list, for example (100,) or (10, 10, 10).
architectures = architectures + []

In [ ]:
rows, models = [], {}
for arch in architectures:
    model = MLPRegressor(hidden_layer_sizes=arch, learning_rate_init=0.01, max_iter=500, random_state=42)
    model.fit(X_train_s, y_train)
    mae_train, mae_test = evaluate(model, f"MLP {arch}")
    n_par = sum(W.size for W in model.coefs_) + sum(b.size for b in model.intercepts_)
    rows.append({"architecture": str(arch), "parameters": n_par, "epochs": model.n_iter_,
                 "train MAE": round(mae_train, 2), "test MAE": round(mae_test, 2)})
    models[str(arch)] = model

results = pd.DataFrame(rows)
results

In [ ]:
lin_test_mae = mean_absolute_error(y_test, lin.predict(X_test_s))
pos = np.arange(len(results))
plt.figure(figsize=(7, 3.8))
plt.bar(pos - 0.2, results["train MAE"], width=0.4, label="train MAE", color="#378DBD")
plt.bar(pos + 0.2, results["test MAE"], width=0.4, label="test MAE", color="#0C234B")
plt.axhline(lin_test_mae, color="#AB0520", linestyle="--", label=f"linear regression, test ({lin_test_mae:.2f})")
plt.xticks(pos, results["architecture"])
plt.xlabel("Hidden layers (neurons per layer)")
plt.ylabel("Mean absolute error (kWh/m²)")
plt.title("Heating load: error vs. network size")
plt.ylim(0, 1.35 * max(results["test MAE"].max(), lin_test_mae))   # leave room for the legend
plt.legend(loc="upper center", ncol=3, fontsize=9)
plt.show()

In [ ]:
best = results.loc[results["test MAE"].idxmin(), "architecture"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, [("Linear regression", lin), (f"MLP {best}", models[best])]):
    ax.scatter(y_test, model.predict(X_test_s), s=12, alpha=0.7, color="#0C234B")
    ax.plot([5, 45], [5, 45], color="#AB0520", linestyle="--", label="perfect prediction")
    ax.set_title(name)
    ax.set_xlabel("Actual heating load (kWh/m²)")
    ax.legend()
axes[0].set_ylabel("Predicted heating load (kWh/m²)")
fig.suptitle("Test buildings: predicted vs. actual")
plt.tight_layout()
plt.show()

**✍️ Question:** 🏠 *Finish at home if you run short of time.* Look at the table and the plots. Did the biggest network always win? Which network would you choose for this task, and why? What does the predicted-vs-actual plot of the linear model show that the MLP fixes?

*Your answer:* 

### 3.6 Overfitting: what if we only had 40 buildings?
Simulations take time, so imagine we had run only **40** of them. We train three models on just those 40 buildings
and test them on the same 154 test buildings as before. `.iloc[:40]` takes the first 40 rows *by position*.

In [ ]:
X_small = X_train_s[:40]
y_small = y_train.iloc[:40]
print(len(X_small), "training buildings")

> 💡 **Hint:** `big.fit(X_small, y_small)`. Then `mean_absolute_error(y_small, big.predict(X_small))` for the training error and `mean_absolute_error(y_test, big.predict(X_test_s))` for the test error.

In [ ]:
# TODO: train a BIG network (two hidden layers of 100 neurons) on only the 40 buildings.
big = MLPRegressor(hidden_layer_sizes=(100, 100), learning_rate_init=0.01, max_iter=2000, random_state=42)
# 1) fit big on X_small, y_small
# 2) compute its MAE on the 40 training buildings and on the test set
mae_small_train = None
mae_small_test = None
print(f"Big network on 40 buildings: train MAE = {mae_small_train}, test MAE = {mae_small_test}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert mae_small_train is not None and mae_small_test is not None, "Compute both MAE values."
assert mae_small_train < mae_small_test, "The training error should be smaller than the test error here."
print("Done. Now compare with smaller models below.")

In [ ]:
small_net = MLPRegressor(hidden_layer_sizes=(5,), learning_rate_init=0.01, max_iter=2000, random_state=42).fit(X_small, y_small)
small_lin = LinearRegression().fit(X_small, y_small)
compare = pd.DataFrame({
    "model (trained on 40 buildings)": ["Linear regression", "MLP (5,)", "MLP (100, 100)"],
    "parameters": [9, 51, sum(W.size for W in big.coefs_) + sum(b.size for b in big.intercepts_)],
    "train MAE": [mean_absolute_error(y_small, m.predict(X_small)) for m in (small_lin, small_net, big)],
    "test MAE": [mean_absolute_error(y_test, m.predict(X_test_s)) for m in (small_lin, small_net, big)],
}).round(2)
compare

**✍️ Question:** The big network has about 11,000 parameters but saw only 40 buildings. Describe what happened to its training and test errors, and explain it in plain words. Give two ways an engineer could reduce this overfitting.

*Your answer:* 

### 3.7 Optional: early stopping
With `early_stopping=True`, scikit-learn holds back 10% of the training data as a **validation set** and stops
training when the validation score has not improved for 10 epochs in a row. It is a common guard against
overfitting. Compare its test MAE with the (50,) network in §3.5: on these noise-free simulated buildings the
full-data network was not overfitting, so early stopping mainly means training on 10% less data and stopping sooner.
It pays off when the validation error would otherwise start to rise — as with the 40-building network above.

In [ ]:
es = MLPRegressor(hidden_layer_sizes=(50,), learning_rate_init=0.01, max_iter=500,
                  early_stopping=True, random_state=42)
es.fit(X_train_s, y_train)
print("Stopped after", es.n_iter_, "epochs")
evaluate(es, "MLP (50,) early stop");

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** A hidden neuron uses ReLU, has weights (2, −1) and bias 1, and receives the input (x₁, x₂) = (1, 4). What does it output? Show the arithmetic.

*Your answer:* 

**2.** In your own words, why does a neural network need a nonlinear activation function such as ReLU?

*Your answer:* 

**3.** A classmate's network has training MAE = 0.1 kWh/m² and test MAE = 3.0 kWh/m². What is going on, and what is one thing they could try?

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.